# Génération des données v6 : enquête ménage complexe

Ce notebook simule le ciblage tel qu'il se fait sur le terrain :

1. **Les enquêteurs collectent** l'enquête ménage avec Kobo : 72 questions (profil, composition, logement, eau et hygiène, santé, alimentation, stratégies d'adaptation, moyens d'existence, protection, assistance).
2. **L'agent applique la grille Excel** de l'ONG (la même que la v5) : colonnes `score_grille_excel` et `decision_grille_excel`.
3. **Une vérification approfondie** (visite à domicile + comité de ciblage) décide si le ménage est éligible : colonne `eligible`. Le comité regarde tout le ménage et combine les réponses entre elles, avec une part de jugement.

Il n'y a pas de GPS ni de données de contexte externes : tout vient de l'enquête ménage. La logique se trouve dans `generateur_ciblage_v6.py`. Les fichiers produits (`data/ciblage_v6.csv`, `data/ciblage_v6.db`) ne sont pas versionnés.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import generateur_ciblage_v6 as gen

In [2]:
d = gen.generer()
gen.construire_base(d, Path("data/ciblage_v6.db"), Path("data/ciblage_v6.csv"))
d.shape

(100000, 79)

In [3]:
d.head()

,id_menage,date_enquete,enqueteur,localite,statut_menage,duree_deplacement_mois,nombre_deplacements_12m,sexe_chef_menage,age_chef_menage,etat_civil_chef,...,incident_securite_3_mois,biens_perdus_crise,membre_disparu_detenu,documents_identite,restriction_mouvements,assistance_3_derniers_mois,type_assistance_recue,score_grille_excel,decision_grille_excel,eligible
0,MEN_000001,2026-08-19,ENQ_19,Rutshuru,resident,NaN,NaN,homme,57,marie,...,non,non,non,oui,non,non,NaN,16,0,0
1,MEN_000002,2026-04-06,ENQ_16,Rwanguba,resident,NaN,NaN,homme,27,marie,...,non,non,non,oui,non,non,NaN,21,0,0
2,MEN_000003,2026-03-07,ENQ_05,Bambo,resident,NaN,NaN,homme,18,marie,...,non,non,non,oui,non,non,NaN,49,1,0
3,MEN_000004,2026-07-07,ENQ_14,Rwanguba,deplace,0.0,3.0,homme,26,marie,...,non,non,non,oui,non,non,NaN,30,0,1
4,MEN_000005,2026-05-14,ENQ_01,Rwanguba,retourne,NaN,5.0,femme,44,marie,...,non,non,non,oui,oui,non,NaN,32,0,0


#### Contrôles de qualité

In [4]:
pd.Series({
    "enfants + personnes âgées <= taille du ménage": ((d.enfants_moins_5 + d.enfants_5_17 + d.personnes_60_plus) <= d.taille_menage).all(),
    "enfants non scolarisés <= enfants de 5 à 17 ans": (d.enfants_non_scolarises <= d.enfants_5_17).all(),
    "type d'assistance renseigné si et seulement si assistance reçue": ((d.assistance_3_derniers_mois == "oui") == d.type_assistance_recue.notna()).all(),
    "durée de déplacement renseignée si et seulement si déplacé": ((d.statut_menage == "deplace") == d.duree_deplacement_mois.notna()).all(),
}, name="OK").to_frame()

,OK
enfants + personnes âgées <= taille du ménage,True
enfants non scolarisés <= enfants de 5 à 17 ans,True
type d'assistance renseigné si et seulement si assistance reçue,True
durée de déplacement renseignée si et seulement si déplacé,True


#### Éligibilité après vérification

In [5]:
val = d.eligible.value_counts()
fig = px.bar(x = val.index.astype(str), y = val.values, color = val.values, text_auto = True, title = "Ménages éligibles après la vérification (1)")
fig.show()

In [6]:
pd.crosstab(d.eligible, d.decision_grille_excel, rownames=['vérification'], colnames=['grille Excel'], normalize = 'all').round(3)

grille Excel,0,1
vérification,,
0,0.612,0.088
1,0.126,0.174
